# Clase 163 — Markov Decision Processes (numpy, ejecutable)

Un **MDP** es la tupla `(S, A, P, R, γ)`. La **ecuación de Bellman** define el valor óptimo
`V*(s) = max_a Σ_s' P(s'|s,a)·[R(s,a,s') + γ·V*(s')]`.

Aquí construimos a mano un **gridworld resbaladizo 4×4** (estilo FrozenLake) con su matriz de
transición `P` y recompensa `R`, y lo resolvemos con **Value Iteration** y **Policy Iteration**
en numpy puro. Todo se ejecuta.

Requiere: `numpy`.

## 1. Definir el MDP a mano: gridworld 4×4 resbaladizo

Grid (S=start, H=hole, G=goal):
```
 0(S)  1     2     3
 4     5(H)  6     7(H)
 8     9    10    11(H)
12(H) 13    14    15(G)
```
Acciones: 0=izq, 1=abajo, 2=der, 3=arriba. Es **resbaladizo**: la acción elegida se ejecuta
con prob 1/3, y con 1/3 cada una de las dos perpendiculares.

In [ ]:
import numpy as np
np.random.seed(0)

nS, nA = 16, 4
holes = {5, 7, 11, 12}
goal = 15

def next_state(s, a):
    row, col = divmod(s, 4)
    if   a == 0: col = max(col - 1, 0)     # izquierda
    elif a == 1: row = min(row + 1, 3)     # abajo
    elif a == 2: col = min(col + 1, 3)     # derecha
    elif a == 3: row = max(row - 1, 0)     # arriba
    return row * 4 + col

# acciones que ocurren al "resbalar": la intencion + las dos perpendiculares
perp = {0: (0, 1, 3), 1: (1, 0, 2), 2: (2, 1, 3), 3: (3, 0, 2)}

P = np.zeros((nS, nA, nS))                  # P[s, a, s']
R = np.zeros((nS, nA, nS))                  # R[s, a, s']
for s in range(nS):
    for a in range(nA):
        if s in holes or s == goal:
            P[s, a, s] = 1.0                # estados terminales: absorbentes
            continue
        for a_real in perp[a]:
            s2 = next_state(s, a_real)
            P[s, a, s2] += 1.0 / 3.0
            if s2 == goal:
                R[s, a, s2] = 1.0           # +1 solo al alcanzar la meta

assert np.allclose(P.sum(axis=2), 1.0)      # cada (s,a) es una distribucion valida
print("MDP construido: P", P.shape, "| suma de probabilidades OK")

## 2. Value Iteration

Aplicamos el operador de Bellman óptimo hasta que `V` deja de cambiar. La convergencia está
garantizada porque es una contracción (factor `γ < 1`).

In [ ]:
def value_iteration(P, R, gamma=0.99, tol=1e-10):
    V = np.zeros(nS)
    for it in range(1, 10001):
        # Q[s,a] = sum_s' P[s,a,s'] * (R[s,a,s'] + gamma * V[s'])
        Q = np.einsum("sap,sap->sa", P, R + gamma * V[None, None, :])
        V_new = Q.max(axis=1)
        if np.max(np.abs(V_new - V)) < tol:
            return V_new, Q.argmax(axis=1), it
        V = V_new
    return V, Q.argmax(axis=1), it

gamma = 0.99
V_star, pi_star, iters = value_iteration(P, R, gamma)
print(f"Value Iteration convergio en {iters} iteraciones")
print("V*  =\n", np.round(V_star.reshape(4, 4), 3))
arrows = np.array(list("<v>^"))
grid_pi = arrows[pi_star].reshape(4, 4)
for s in holes: grid_pi.flat[s] = "H"
grid_pi.flat[goal] = "G"
print("policy* =\n", grid_pi)

## 3. Policy Iteration

Alterna **evaluación** (resolver `V^π`) y **mejora** (`π' = greedy(V^π)`) hasta que la policy
se estabiliza. Suele necesitar menos iteraciones que Value Iteration, aunque cada una es más cara.

In [ ]:
def policy_evaluation(pi, P, R, gamma, tol=1e-10):
    V = np.zeros(nS)
    while True:
        # solo la accion elegida por la policy en cada estado
        P_pi = P[np.arange(nS), pi]                 # (nS, nS)
        R_pi = (P_pi * R[np.arange(nS), pi]).sum(axis=1)
        V_new = R_pi + gamma * (P_pi @ V)
        if np.max(np.abs(V_new - V)) < tol:
            return V_new
        V = V_new

def policy_iteration(P, R, gamma):
    pi = np.zeros(nS, dtype=int)
    for it in range(1, 1001):
        V = policy_evaluation(pi, P, R, gamma)
        Q = np.einsum("sap,sap->sa", P, R + gamma * V[None, None, :])
        pi_new = Q.argmax(axis=1)
        if np.array_equal(pi_new, pi):
            return V, pi_new, it
        pi = pi_new
    return V, pi, it

V_pi, pi_pi, iters_pi = policy_iteration(P, R, gamma)
print(f"Policy Iteration convergio en {iters_pi} iteraciones")
print("misma policy que Value Iteration:", np.array_equal(pi_pi, pi_star))
print("misma V* (aprox):", np.allclose(V_pi, V_star, atol=1e-6))

## 4. Evaluar la policy con simulación Monte Carlo

Simulamos episodios muestreando el próximo estado desde `P` y medimos el **success rate**
(fracción de episodios que alcanzan la meta).

In [ ]:
def rollout(pi, P, max_steps=100, seed=0):
    rng = np.random.default_rng(seed)
    s = 0                                            # start
    for _ in range(max_steps):
        s = rng.choice(nS, p=P[s, pi[s]])
        if s == goal:  return True
        if s in holes: return False
    return False

n = 2000
success = np.mean([rollout(pi_star, P, seed=i) for i in range(n)])
print(f"success rate de la policy optima ({n} episodios): {success:.3f}")
print("(en FrozenLake resbaladizo, una buena policy logra >= 0.70)")

## 5. Limitación y motivación

Value/Policy Iteration **requieren conocer `P` y `R`** (el modelo del MDP) y no escalan a
espacios de estados grandes o continuos. En entornos reales rara vez conocemos `P`: eso motiva
los métodos **model-free** como Q-learning (clase 164).

## 6. Variante determinista (ejecutable)

Si el grid NO resbala, cada `(s,a)` lleva con certeza a un único estado. La policy óptima traza
el camino más corto a la meta evitando huecos, y el success rate llega a 1.0.

In [ ]:
P_det = np.zeros((nS, nA, nS))
R_det = np.zeros((nS, nA, nS))
for s in range(nS):
    for a in range(nA):
        if s in holes or s == goal:
            P_det[s, a, s] = 1.0
            continue
        s2 = next_state(s, a)                # sin resbalar: accion determinista
        P_det[s, a, s2] = 1.0
        if s2 == goal:
            R_det[s, a, s2] = 1.0

V_det, pi_det, it_det = value_iteration(P_det, R_det, gamma)
print(f"determinista: VI convergio en {it_det} iteraciones")
sr = np.mean([rollout(pi_det, P_det, seed=i) for i in range(500)])
print(f"success rate (determinista): {sr:.3f}")

## Ejercicios

1. Cambiar `γ` a 0.9 y 0.999; observar cómo se modifican `V*` y la policy.
2. Volver el gridworld **determinista** (`P[s,a,next_state(s,a)] = 1`) y verificar que la policy
   traza el camino más corto evitando huecos.
3. Añadir una penalización `-0.01` por paso en `R` y comprobar que la policy se vuelve más directa.
4. Comparar el número de iteraciones de Value Iteration vs Policy Iteration al variar `γ`.

## Conclusiones

- Un MDP `(S, A, P, R, γ)` formaliza el problema de RL cuando el modelo es conocido.
- Value Iteration aplica el operador de Bellman óptimo hasta converger (es una contracción).
- Policy Iteration alterna evaluación y mejora; converge en pocas iteraciones a la misma `π*`.
- Ambos exigen conocer `P` y `R`, lo que casi nunca ocurre: de ahí los métodos model-free.